In [1]:
import threading
import time

In [2]:
num_threads = 5
iteratoin_in_one_thread = 100

In [28]:
counter = 0   # just a variable

In [4]:
def f():
    global counter

    for i in range(iteratoin_in_one_thread):
        # read from counter
        v = counter
        time.sleep(0.000000000000000001)  # Do some stuff
        v += 1

        # write back to counter after some time
        counter = v

In [14]:
def run_experiment():
    global counter
    threads = []

    for i in range(num_threads):
        t = threading.Thread(target=f)
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print(f"Calculated value: {counter}")
    print(f"Expected value: {num_threads * iteratoin_in_one_thread}")                             

In [15]:
run_experiment()

Calculated value: 102
Expected value: 500


# Expectation vs Reality
This is what we expect will happen. 

In [16]:
counter = 0 

v1 = counter 
v1 += 1 
counter = v1 

v2 = counter 
v2 += 1 
counter = v2 


print(counter) 

2


This is what really happens. 

In [18]:
counter = 0 

# --- thread 1 start 
v1 = counter 
v1 += 1 
# ---- break 

# --- thread 2 start 
v2 = counter 
v2 = 100 
counter = v2 
# ---- thread 2 done 

# ---- back to thread 1 
counter = v1 
# ---- thread 1 done 

print(counter) 

1


# Synchronization

What we need is some way of ensuring that <strong>critical sections</strong> are executed all in one go. 

In [21]:
lock = threading.Lock()

In [22]:
def f():
    global counter

    for i in range(iteratoin_in_one_thread):
        lock.acquire()  # begin the critical section
        
        # read from counter
        v = counter
        time.sleep(0.000000000000000001)  # Do some stuff
        v += 1

        # write back to counter after some time
        counter = v

        lock.release()  # end the critical section

In [25]:
run_experiment()

Calculated value: 500
Expected value: 500


But we will surely forget to release the lock ... just as we would forget to close open files. Well, context managers to the rescue! 

In [26]:
def f():
    global counter, lock

    for i in range(iteratoin_in_one_thread):
        with lock:      # it will release lock automatically
            # read from counter
            v = counter
            time.sleep(0.000000000000000001)  # Do some stuff
            v += 1
    
            # write back to counter after some time
            counter = v

In [29]:
run_experiment()

Calculated value: 500
Expected value: 500


You do have to figure out which part you <strong>don't</strong> want breaks in! 

It's much more common to figure out which parts you <strong>do</strong> want the thread suspended at! 